# Setting up Controller and Event Analyst

Hello! This tutorial will show you how to set up `Ralph` to perform the tasks for you!

`Ralph` can:
* pre-process the light curves to make sure the fitting process runs smoothly;
* fit ongoing and finished gravitational microlensing events;
* create a color-magnitude diagram for all found best-fitting models.

`Ralph` is pretty flexible, and you can and may turn on and off certain modules.

## Overview

Ralph has two basic classes, a Controller and an Analyst.

The **Controller** manages the workflow for all events you'd like to analyze. It controls how many events are handled at once,
where the inputs and output files go, and how, where, at what level the logs are created. For each event you'd like to analyze,
the Controller will launch an **Event Analyst**.

The **Event Analyst** controls the analysis work flow for a single event. It handles sub-analysts (Light Curve, Fot, and CMD). Event Analyst launches respective sub-analysts based on the Event Analyst configuration for each event. It knows where to store the outputs and the logs.
For each event you may or may not confugure your Event Analyst differently. If you have a set of events, each with a varying data quality, you can set up their pre-processing or fitting procedures differently.
Or, if you have color-magnitude diagram information for only some of the events, or only some of the surveys, you can set up too!
Each event can be treated individually!

The **Event Analyst** manages three types of Analysts:
* the **Light Curve Analyst** -- it's handling the pre-processing of the light curves, to make sure no invalid entries prevent
running the fitting process smoothly;
* the **Fit Analyst** -- it's handling the fitting procedure;
* the **CMD Analyst** -- it's handling the creation of the color-magnitude diagram for each event and catalog you have specified.


## Configuring the Controller

Okay! Let's start with something simple and set up our controller to run on a single event.
First we have to let `Ralph` know how do we set up the Controller.
You can pass a YAML or JSON file with the configuration, but you can also pass it as a dictionary.
We will create a dictionary, to make it easier in this tutorial.

In [1]:
import os

# Here we set up the path to the tutorial folder, to ensure that the example executes correctly
current_path = os.getcwd()
os.chdir("../../")
ralph_home_dir = os.getcwd()

In [2]:
ralph_home_dir

'/home/kasia/Documents/microlensing_fitting/ralph'

In [3]:
config = {
    # Name of the python compiler you want to use for this run.
    # It can be just "python" to use a default compiler, but you
    # can also specify python3, or python3.10, python3.12, etc.
    "python_compiler": "python",
    # Group processing limit refers to how many events will be
    # analyzed at the same time. It is limited by how many cores
    # are available on your machine.
    "group_processing_limit": 2,
    # What file format (YAML or JSON) will you use for your configuration
    # files for the Event Analysts.
    "config_type": "json",
    # Where will the outputs be saved and where Event Analysts
    # should look for the configuration files for each event.
    # You can pass it as a string, or use the power of the os
    # package, when creating a dictionary.
    "events_path": os.path.join(ralph_home_dir, "examples", "controller"),
    # Because Ralph uses the subprocess package, it requires specific set up.
    # You have to specify what is the path of your Ralph's repository location
    # of the Analysts' source files.
    "software_dir": os.path.join(ralph_home_dir, "src", "ralph", "analyst"),
    # Ralph uses logging.Logger instance to handle logging statements for
    # Controller. If log_stream is set to "True", it can be streamed to
    # the standard output, and then captured with for example Kubernetes.
    # If set to false, the log will be saved to the location specified
    # in log_location.
    "log_stream": True,
    # Where you'd like to save the file with controller logs. The file
    # will appear in the log_location directory, under the name
    # "controller.log"
    "log_location": os.path.join(ralph_home_dir, "examples", "controller"),
    # How verbose the log statements should be? There are three levels
    # available: "error" (least verbose), "info" (medium), "debug" (very
    # verbose).
    "log_level": "info",
}

On top of the configuration information, `Ralph` also needs to know which events you'd like to have analyzed.
For the sake of simplicity, we will analyze only one event.

These event names will correspond to folder names inside the `events_path` location.

<!-- * If you pass the Event Analyst configuration as dictionaries created on the fly,
`Ralph` will create folders with event names inside the `events_path` directory. -->
* If you pass the Event Analyst configuration as files, they have to be located inside the
`events_path` folder, in a folder with the same name as your event name, under `config.yaml`
or `config.json` (whichever file format you specified in the Controller configuration under
`config_type`).

In [4]:
# We will analyze only one event from the OGLE survey
event_list = [
    "OGLE_2016_BLG_1395",
]

Great! Now we have to set up our Event Analyst configuration and input files, before we can move forward.

## Configuring the Event Analyst

Let's say you only want to perform some basic fitting with `Ralph` for your event. 
We will show you what the minimal information is necessary to launch an event fitting.

This configuration file can, but doesn't have to include the following keywords:
* `lc_analyst` - it should hold a dictionary with the Light Curve Analyst set up.
If you leave this as an empty dictionary, it will launch the Light Curve Analyst with the default setup.
You can also drop this keyword completely, to not run the Light Curve Analyst at all, but it's not recommended
if you want to run the Fit Analyst.
* `fit_analyst` - it should hold a dictionary with the Fit Analyst set up. You can leave it empty, and it will
use the default setup for fitting. The default is `pyLIMA` as a fitting package, and
the [**Trust Region Reflective**](https://github.com/ebachelet/pyLIMA/blob/master/pyLIMA/fits/TRF_fit.py) fitting
routine for all types of models, except the initial one, where the [**Differential Evolution**](https://github.com/ebachelet/pyLIMA/blob/master/pyLIMA/fits/DE_fit.py) is used.
You can also drop this keyword completely, to not run the Fit Analyst at all,
* `cmd_analyst` - it should hold a dictionary with the CMD Analyst set up. Unlike the two previous Analysts, it doesn't have a default setup.
If you'd like to run the CMD Analyst for an event, you **have** to provide the configuration.

### The bare minimum

In this example, we will run the Light Curve Analyst with the default setup, and we will customize the Fit Analyst (in a moment).

In [5]:
event_analyst_config = {
    # Event name, should match an event name provided in "event_list".
    "event_name": "OGLE_2016_BLG_1395", # Mandatory
    # Event Right Ascension, in degrees
    "ra": 271.1194, # Mandatory
    # Event Declination, in degrees
    "dec": -29.8162, # Mandatory
    # A dictionary with the Light Curve Analyst setup.
    # We will leave it empty, to run with the default setup.
    "lc_analyst": {
        # "default": "setup",  # Optional field
    },
    # A dictionary with the Fit Analyst setup.
    # We will leave it empty for now, because it's the most complex one.
    "fit_analyst": {}, # Optional field
    # A list of dictionaries with light curves.
    "light_curves": [ # Mandatory
        {
            # Survey or telescope name, which was used to obtain the data.
            "survey": "OGLE", # Mandatory
            # Band (or filer) in which the data was obtained.
            "band": "I", # Mandatory
            # Path to a location where the light curve is stored.
            "path": os.path.join(
                ralph_home_dir, "examples", "example_light_curve_OB161395_I.dat"
            ) # Mandatory
        },
    ]
}

The file with the light curve has to contain three columns:
* first column with Julian Day of each taken data point;
* second column with magnitude of each taken data point;
* third column with uncertianity of magnitude for each taken data point;
Let's have a look at the example:

In [6]:
import numpy as np

data = np.loadtxt(os.path.join(ralph_home_dir, "examples", "example_light_curve_OB161395_I.dat"))
data

array([[2.45526088e+06, 1.49340000e+01, 5.05800000e-03],
       [2.45526181e+06, 1.49360000e+01, 5.06200000e-03],
       [2.45526285e+06, 1.49430000e+01, 8.50800000e-03],
       ...,
       [2.45892387e+06, 1.49270000e+01, 5.04300000e-03],
       [2.45892389e+06, 1.49330000e+01, 5.05600000e-03],
       [2.45892389e+06, 1.49320000e+01, 5.05400000e-03]], shape=(2780, 3))

### Fit Analyst configuration

Now let's focus on the configuration of the Fit Analyst. It can get quite complex, so we will go step by step.

Fit Analyst has four major keywords:
* `ongoing_magnification_thershold` (default value: 1.05) -- threshold for magnification, if the magnification of the last data point for a simple point source-point lens model (PSPL) is larger, the event is considered as ongoing;
* `ongoing_amplitude_thershold` (default value: 1.0 mag) -- thershold for amplitude from baseline of the last data point, if the amplitude is larger, the event is considered as ongoing;
* `time_of_peak_bin_size` (default value: 2.0 days) -- bin size for a function that is looking for the best approximation of time of peak for the first model fit;
* `model_fit_configuration` -- a dictionary with fit configuration, we will come back to it.

In [7]:
event_analyst_config["fit_analyst"] = {
    # Threshold for PSPL model magnification
    "ongoing_magnification_thershold" : 1.10,
    # Threshold for amplitude
    "ongoing_amplitude_thershold": 1.0,
    # Bin size when looking for the time of peak
    "time_of_peak_bin_size": 1.0,
    # Configuration of each fit for specific models
    "model_fit_configuration" : {}
}

You can configure the Fit Analyst for the following supported model types:

* `PSPL_no_blend_no_piE` -- point source point lens model without blending and microlensing parallax effect;
* `PSPL_blend_no_piE` -- point source point lens model with blending and without microlensing parallax effect;
* `PSPL_blend_piE` -- point source point lens model with blending and microlensing parallax effect;
* `PSPL_no_blend_piE` -- point source point lens model without blending and with microlensing parallax effect;

For each model type you can provide the a dictionary with the following keywords:
* `fitting_package` (default value: pyLIMA) -- the name of the fitting package you want to use for this model; currently only pyLIMA is supported;
* `fitting_method` (default value: TRF) -- the name of the fitting method supported by the `fitting_package` for this model, available options: TRF (Trust Reflective Function), DE (Differential Evolution);
* `boundaries` -- a dictionary with list of lower and upper boundry for each microlensing model parameter (`t0`, `u0`, `tE`, `piEN`, `piEE`) you'd like to modify.

 :warning:  :warning:  :warning: Currently `Ralph` supports only pyLIMA as a fitting package. :warning: :warning: :warning:

Below we will set programatically the fitting package for each model, and then specify more parameters for the `PSPL_blend_piE` model.

In [8]:
model_list = [
    "PSPL_no_blend_no_piE",
    "PSPL_blend_no_piE",
    "PSPL_blend_piE",
    "PSPL_no_blend_piE",
]

for model in model_list:
    if model != "PSPL_blend_piE":
        event_analyst_config["fit_analyst"]["model_fit_configuration"][model] =  {
            "fitting_package": "pyLIMA"
        }
    else:
        event_analyst_config["fit_analyst"]["model_fit_configuration"][model] =  {
            "fitting_package": "pyLIMA",
            "fitting_method": "DE", 
            "boundaries": {
                "u0": [ -2.0, 2.0 ],
                "piEN": [ -1.0, 1.0 ],
                "piEE": [ -1.0, 1.0 ],
            }
        }

Controller looks for Event Analyst configuration in a very specific way. It checks the directory specified in the Controller `config["events_path"]`. There it looks for subdirectories named like the events passed in the `event_list` variable of the Controller. Such subdirectory has to contain a file called `config.json` with the Event Analyst configuration.

To pass our Event Analyst configuration dictionary to Controller, we will have to turn it into a JSON, and then save it in a location where the Controller will look for files.

In [9]:
import json

config_path = os.path.join(
    ralph_home_dir, "examples", "controller", "OGLE_2016_BLG_1395", "config.json"
)
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(event_analyst_config, f, ensure_ascii=False, indent=4)

## Initializing the Controller

Now we can initialize the Controller.

In [10]:
from ralph.controller.controller import Controller

controller = Controller(event_list, config_dict=config)

2026-06-11 15:15:26.200 - INFO - Processing started. Opened log.


Let's look up the Controller configuration:

In [11]:
for key in controller.config:
    print(key, ":", controller.config[key])

python_compiler : python
group_processing_limit : 2
config_type : json
events_path : /home/kasia/Documents/microlensing_fitting/ralph/examples/controller
software_dir : /home/kasia/Documents/microlensing_fitting/ralph/src/ralph/analyst
log_stream : True
log_location : /home/kasia/Documents/microlensing_fitting/ralph/examples/controller
log_level : info


Now, we can launch the Controller, and see what it's doing through the logs.

In [12]:
controller.launch_analysts()

2026-06-11 15:15:27.978 - INFO - Controller: Start processing.
2026-06-11 15:15:27.978 - INFO - Controller: Commands created. Spawning processes.
2026-06-11 15:15:27.998 - INFO - About to start subprocess for event: 
2026-06-11 15:15:28.001 - INFO - Command: ['python', '/home/kasia/Documents/microlensing_fitting/ralph/src/ralph/analyst/event_analyst.py', '--event_name', 'OGLE_2016_BLG_1395', '--analyst_path', '/home/kasia/Documents/microlensing_fitting/ralph/examples/controller/OGLE_2016_BLG_1395/', '--log_level', 'info', '--stream', 'True']
-------------------------------------
/home/kasia/Documents/microlensing_fitting/ralph/examples/controller/OGLE_2016_BLG_1395/
/home/kasia/Documents/microlensing_fitting/ralph/examples/controller/OGLE_2016_BLG_1395/
2026-06-11 15:15:30.160 - INFO - Processing started. Opened log.
2026-06-11 15:15:30.161 - INFO - -------------------------------------------
2026-06-11 15:15:30.161 - INFO - Event Analyst: Analyzing event OGLE_2016_BLG_1395
2026-06-11 

/home/kasia/Documents/microlensing_fitting/.venv/lib/python3.12/site-packages/pyLIMA/fits/stats.py:48: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  AD_stat, AD_critical_values, AD_significance_levels = ss.anderson(sample)


2026-06-11 15:15:32.762 - INFO - Fit Analyst: Plotting finished.
2026-06-11 15:15:32.798 - INFO - Fit Analyst:  Finished fitting.
2026-06-11 15:15:32.798 - INFO - Fit Analyst: Identify ongoing event.
2026-06-11 15:15:32.798 - INFO - Fit Analyst: Performing a finished event fit.
2026-06-11 15:15:32.799 - INFO - Fit Analyst: Starting finished event fit.
2026-06-11 15:15:32.799 - INFO - Fit Analyst: Finding PSPL starting parameters.
2026-06-11 15:15:32.799 - INFO - Fit Analyst:Performing PSPL with blend fit.
2026-06-11 15:15:32.799 - INFO - Fit Analyst: Using fitting setup specified by the User.
check_event  : Everything looks fine...
2026-06-11 15:15:32.829 - INFO - Fit Analyst -- pyLIMA: Fitting without microlensing parallax.
2026-06-11 15:15:32.830 - INFO - Fit Analyst -- pyLIMA: Using default fitting method (TRF).
2026-06-11 15:15:32.854 - INFO - Fit Analyst -- pyLIMA: Using boundaries default for ralph.
2026-06-11 15:15:32.854 - INFO - Fit Analyst -- pyLIMA: Adding starting parameter

/home/kasia/Documents/microlensing_fitting/.venv/lib/python3.12/site-packages/pyLIMA/fits/stats.py:48: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  AD_stat, AD_critical_values, AD_significance_levels = ss.anderson(sample)


2026-06-11 15:15:35.102 - INFO - Fit Analyst: Plotting finished.
2026-06-11 15:15:35.135 - INFO - Fit Analyst: Finished fitting PSPL with blend fit.
2026-06-11 15:15:35.135 - INFO - Fit Analyst: Performing PSPL+piE fit.
2026-06-11 15:15:35.135 - INFO - Fit Analyst: Starting fitting model PSPL_blend_piE_p
2026-06-11 15:15:35.135 - INFO - Fit Analyst: Using fitting setup specified by the User.
check_event  : Everything looks fine...
2026-06-11 15:15:35.168 - INFO - Fit Analyst -- pyLIMA: Fitting with microlensing parallax.
Parallax(Full) estimated for the telescope OGLE_I: SUCCESS
2026-06-11 15:15:36.046 - INFO - Fit Analyst -- pyLIMA: Fitting method: DE.
2026-06-11 15:15:36.075 - INFO - Fit Analyst -- pyLIMA: Using boundaries passed by the User.
2026-06-11 15:15:36.075 - INFO - Fit Analyst -- pyLIMA: Adding starting parameters:
2026-06-11 15:15:36.075 - INFO - Fit Analyst -- pyLIMA: Adding starting parameters: t0 = 2457721.453
2026-06-11 15:15:36.075 - INFO - Fit Analyst -- pyLIMA: Addi

/home/kasia/Documents/microlensing_fitting/.venv/lib/python3.12/site-packages/pyLIMA/toolbox/brightness_transformation.py:39: RuntimeWarning: invalid value encountered in log10
  mag = ZERO_POINT - 2.5 * np.log10(flux)
/home/kasia/Documents/microlensing_fitting/.venv/lib/python3.12/site-packages/pyLIMA/fits/stats.py:48: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  AD_stat, AD_critical_values, AD_significance_levels = 

Parallax(Full) estimated for the telescope OGLE_I: SUCCESS
2026-06-11 15:15:59.372 - INFO - Fit Analyst: Plotting finished.
2026-06-11 15:15:59.372 - INFO - Fit Analyst:  Finished fitting model PSPL_blend_piE_p
2026-06-11 15:15:59.372 - INFO - Fit Analyst: Starting fitting model PSPL_blend_piE_n
2026-06-11 15:15:59.372 - INFO - Fit Analyst: Using fitting setup specified by the User.
check_event  : Everything looks fine...
2026-06-11 15:15:59.404 - INFO - Fit Analyst -- pyLIMA: Fitting with microlensing parallax.
Parallax(Full) estimated for the telescope OGLE_I: SUCCESS
2026-06-11 15:15:59.615 - INFO - Fit Analyst -- pyLIMA: Fitting method: DE.
2026-06-11 15:15:59.641 - INFO - Fit Analyst -- pyLIMA: Using boundaries passed by the User.
2026-06-11 15:15:59.641 - INFO - Fit Analyst -- pyLIMA: Adding starting parameters:
2026-06-11 15:15:59.641 - INFO - Fit Analyst -- pyLIMA: Adding starting parameters: t0 = 2456678.559
2026-06-11 15:15:59.642 - INFO - Fit Analyst -- pyLIMA: Adding starti

/home/kasia/Documents/microlensing_fitting/.venv/lib/python3.12/site-packages/pyLIMA/toolbox/brightness_transformation.py:39: RuntimeWarning: invalid value encountered in log10
  mag = ZERO_POINT - 2.5 * np.log10(flux)
/home/kasia/Documents/microlensing_fitting/.venv/lib/python3.12/site-packages/pyLIMA/fits/stats.py:48: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  AD_stat, AD_critical_values, AD_significance_levels = 

Parallax(Full) estimated for the telescope OGLE_I: SUCCESS
2026-06-11 15:17:18.834 - INFO - Fit Analyst: Plotting finished.
2026-06-11 15:17:18.834 - INFO - Fit Analyst:  Finished fitting model PSPL_blend_piE_n
2026-06-11 15:17:18.843 - INFO - Event Analyst: Processing finished.
2026-06-11 15:17:18.843 - INFO - -------------------------------------------
2026-06-11 15:17:18.843 - INFO - Processing complete.

2026-06-11 15:17:19.366 - INFO - Controller: Processing finished.
2026-06-11 15:17:19.366 - INFO - Processing complete.



After running the Controller, you should see the following files in the folder with the event:

In [14]:
event_dir = os.path.join(ralph_home_dir, "examples", "controller", "OGLE_2016_BLG_1395")

for file in os.listdir(event_dir):
    print(file)

config.json
fit_stats.txt
.ipynb_checkpoints
fit_results.json


You can look up fitting results in `fit_results.json`, while `fit_stats.txt` contains useful statistics you can use to judge which model is the best.